# EDA — Système de Prédiction de la Réussite Académique
**Département ISIC — Ingénierie des Systèmes d'Information et de Communication**

Ce notebook effectue une analyse exploratoire complète des données étudiantes:
- Chargement et nettoyage des données
- Distribution des notes et absences
- Matrice de corrélation
- Identification des facteurs clés de réussite

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

print('Librairies chargees avec succes!')

## 1. Chargement des données

In [ ]:
from src.preprocessing.data_pipeline import (
    load_and_rename, clean_data, handle_missing, create_features, create_target_binary
)

raw_path = '../data/raw/isic.xlsx'
df_raw = load_and_rename(raw_path)
df = clean_data(df_raw.copy())
df = handle_missing(df)
df = create_features(df)
df = create_target_binary(df)

print(f'Dataset: {df.shape[0]} étudiants, {df.shape[1]} colonnes')
df.head()

## 2. Aperçu général

In [ ]:
print('=== Statistiques descriptives ===')
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe().round(2)

In [ ]:
print('=== Valeurs manquantes ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'Aucune valeur manquante!')

## 3. Distribution des notes (S1 et S2)

In [ ]:
# Colonnes de notes
notes_S1 = ['Mathematiques_1', 'Algorithmique_Prog', 'Architecture_Ord',
            'Electronique_Num', 'Reseaux_Info_1', 'Anglais_Tech_1', 'Francais_Pro_1']
notes_S2 = ['Mathematiques_2', 'Structures_Donnees', 'Systemes_Exploitation',
            'Bases_Donnees', 'Reseaux_Info_2', 'Anglais_Tech_2', 'Francais_Pro_2', 'PFA_2']

notes_S1_present = [c for c in notes_S1 if c in df.columns]
notes_S2_present = [c for c in notes_S2 if c in df.columns]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# S1
df[notes_S1_present].plot(kind='box', ax=axes[0], vert=False, patch_artist=True)
axes[0].set_title('Distribution des notes — Semestre 1', fontsize=14, fontweight='bold')
axes[0].axvline(x=10, color='red', linestyle='--', linewidth=1.5, label='Seuil 10')
axes[0].legend()

# S2
df[notes_S2_present].plot(kind='box', ax=axes[1], vert=False, patch_artist=True)
axes[1].set_title('Distribution des notes — Semestre 2', fontsize=14, fontweight='bold')
axes[1].axvline(x=10, color='red', linestyle='--', linewidth=1.5, label='Seuil 10')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/boxplots_notes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegarde: docs/boxplots_notes.png')

## 4. Distribution de la Moyenne Annuelle et classification

In [ ]:
if 'Moyenne_Annuelle' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogramme
    axes[0].hist(df['Moyenne_Annuelle'].dropna(), bins=15, color='#4e79a7', edgecolor='white', alpha=0.85)
    axes[0].axvline(x=10, color='red', linestyle='--', linewidth=2, label='Seuil 10 (reussite)')
    axes[0].axvline(x=df['Moyenne_Annuelle'].median(), color='orange', linestyle='--',
                    linewidth=2, label=f'Mediane ({df["Moyenne_Annuelle"].median():.1f})')
    axes[0].set_xlabel('Moyenne Annuelle (/20)')
    axes[0].set_ylabel('Nombre d etudiants')
    axes[0].set_title('Distribution de la Moyenne Annuelle', fontweight='bold')
    axes[0].legend()

    # Camembert VERT/JAUNE/ROUGE
    def get_couleur(note):
        if note >= 14: return 'VERT'
        elif note >= 10: return 'JAUNE'
        else: return 'ROUGE'

    df['Couleur'] = df['Moyenne_Annuelle'].apply(get_couleur)
    counts = df['Couleur'].value_counts()
    colors_map = {'VERT': '#2ecc71', 'JAUNE': '#f39c12', 'ROUGE': '#e74c3c'}
    clrs = [colors_map.get(c, 'gray') for c in counts.index]

    axes[1].pie(counts.values, labels=counts.index, colors=clrs, autopct='%1.1f%%',
                startangle=90, textprops={'fontsize': 12})
    axes[1].set_title('Repartition Vert / Jaune / Rouge', fontweight='bold')

    plt.tight_layout()
    plt.savefig('../docs/distribution_moyenne.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(df['Couleur'].value_counts())

## 5. Matrice de Corrélation

In [ ]:
# Selectionner uniquement les colonnes numeriques pertinentes
corr_cols = [c for c in [
    'Absences_S1', 'Moyenne_S1', 'Absences_S2', 'Moyenne_S2',
    'Mathematiques_1', 'Algorithmique_Prog', 'Mathematiques_2',
    'Structures_Donnees', 'PFA_2', 'Modules_Non_Valides',
    'Total_Absences', 'Progression', 'Moyenne_Annuelle'
] if c in df.columns]

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matrice de Correlation — Variables Académiques', fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../docs/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Matrice de correlation sauvegardee!')

## 6. Impact des Absences sur la Réussite

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'Absences_S1' in df.columns and 'Moyenne_S1' in df.columns:
    axes[0].scatter(df['Absences_S1'], df['Moyenne_S1'],
                    alpha=0.7, color='#4e79a7', edgecolor='white', s=100)
    z = np.polyfit(df['Absences_S1'].fillna(0), df['Moyenne_S1'].fillna(0), 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['Absences_S1'].min(), df['Absences_S1'].max(), 100)
    axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label='Tendance')
    axes[0].set_xlabel('Absences S1')
    axes[0].set_ylabel('Moyenne S1 (/20)')
    axes[0].set_title('Absences vs Moyenne S1', fontweight='bold')
    axes[0].legend()

if 'Total_Absences' in df.columns and 'Moyenne_Annuelle' in df.columns:
    axes[1].scatter(df['Total_Absences'], df['Moyenne_Annuelle'],
                    alpha=0.7, color='#e15759', edgecolor='white', s=100)
    z = np.polyfit(df['Total_Absences'].fillna(0), df['Moyenne_Annuelle'].fillna(0), 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['Total_Absences'].min(), df['Total_Absences'].max(), 100)
    axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Tendance')
    axes[1].set_xlabel('Total Absences (S1 + S2)')
    axes[1].set_ylabel('Moyenne Annuelle (/20)')
    axes[1].set_title('Total Absences vs Moyenne Annuelle', fontweight='bold')
    axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/absences_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (corrélation avec la Moyenne Annuelle)

In [ ]:
if 'Moyenne_Annuelle' in df.columns:
    feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in ['Moyenne_Annuelle', 'Reussite']]
    correlations = df[feature_cols].corrwith(df['Moyenne_Annuelle']).abs().sort_values(ascending=False)

    plt.figure(figsize=(10, 7))
    colors = ['#2ecc71' if v > 0.7 else '#f39c12' if v > 0.4 else '#e74c3c'
              for v in correlations.values]
    bars = plt.barh(correlations.index, correlations.values, color=colors, edgecolor='white', alpha=0.85)
    plt.axvline(x=0.5, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='Seuil 0.5')
    plt.xlabel('Correlation avec Moyenne Annuelle (valeur absolue)', fontsize=12)
    plt.title('Importance des Features — Correlation avec la Reussite', fontsize=14, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.savefig('../docs/feature_importance_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('Top 5 features les plus correlees avec la Moyenne Annuelle:')
    print(correlations.head(5).to_string())

## 8. Progression S1 → S2

In [ ]:
if 'Moyenne_S1' in df.columns and 'Moyenne_S2' in df.columns:
    plt.figure(figsize=(10, 6))
    for _, row in df.iterrows():
        color = '#2ecc71' if row.get('Progression', 0) >= 0 else '#e74c3c'
        plt.plot([1, 2], [row['Moyenne_S1'], row['Moyenne_S2']],
                 'o-', color=color, alpha=0.5, linewidth=1.5, markersize=6)

    # Moyennes globales
    plt.plot([1, 2], [df['Moyenne_S1'].mean(), df['Moyenne_S2'].mean()],
             'o-', color='black', linewidth=4, markersize=12, label='Moyenne promotion', zorder=5)

    plt.xticks([1, 2], ['Semestre 1', 'Semestre 2'], fontsize=13)
    plt.ylabel('Moyenne (/20)', fontsize=12)
    plt.title('Progression des etudiants S1 -> S2', fontsize=14, fontweight='bold')
    prog_pos = (df['Progression'] >= 0).sum()
    prog_neg = (df['Progression'] < 0).sum()
    from matplotlib.patches import Patch
    plt.legend(handles=[
        Patch(color='#2ecc71', label=f'Progression ({prog_pos} etudiants)'),
        Patch(color='#e74c3c', label=f'Regression ({prog_neg} etudiants)'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Moyenne promotion')
    ], fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../docs/progression_s1_s2.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Moyenne S1: {df["Moyenne_S1"].mean():.2f} | Moyenne S2: {df["Moyenne_S2"].mean():.2f}')
    print(f'Progression moyenne: {df["Progression"].mean():+.2f}')

## 9. Résumé de l'EDA

In [ ]:
print('=' * 60)
print('RESUME DE L EDA — ISIC 1ere Annee')
print('=' * 60)
print(f'Nombre d etudiants analysés: {len(df)}')
if 'Moyenne_Annuelle' in df.columns:
    print(f'Moyenne annuelle moyenne: {df["Moyenne_Annuelle"].mean():.2f}/20')
    print(f'Note minimale: {df["Moyenne_Annuelle"].min():.2f}  |  Note maximale: {df["Moyenne_Annuelle"].max():.2f}')
if 'Total_Absences' in df.columns:
    print(f'Absences totales moyennes: {df["Total_Absences"].mean():.1f} heures')
if 'Redoublant' in df.columns:
    print(f'Redoublants: {df["Redoublant"].sum()} / {len(df)}')
print()
print('Graphiques generes dans docs/:')
for f in ['boxplots_notes.png', 'distribution_moyenne.png', 'correlation_matrix.png',
          'absences_impact.png', 'feature_importance_correlation.png', 'progression_s1_s2.png']:
    print(f'  - {f}')